# TRABALHO 3: REGRESSÃO LOGÍSTICA E SVM
**Integrantes:** Julia Welter, Pedro Pacheco, Pedro Albuquerque, Tiago Melo

In [ ]:
import pickle
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, precision_score, recall_score

import utils

# PARTE 3: Regressão Logística

### Questões 1-6: Base 'risco_credito.pkl'

In [ ]:
# 1. Carregar base
df_pequeno = utils.RiscoCredito.carregar('risco_credito.pkl')

# 2. Encoder e remover 'moderado'
df_pequeno = utils.RiscoCredito.remover_moderado(df_pequeno)
X_p = utils.RiscoCredito.encode_previsores(df_pequeno)
y_p, le_p = utils.RiscoCredito.encode_classe(df_pequeno)

# 3. Treinar Regressão Logística (random_state=1)
modelo_lr_p = LogisticRegression(random_state=1)
modelo_lr_p.fit(X_p, y_p)

# 4. Intercepto (B0)
print(f"4. Intercepto (B0): {modelo_lr_p.intercept_[0]:.8f}")

# 5. Coeficientes
print(f"5. Coeficientes: {modelo_lr_p.coef_}")

# 6. Predict
cliente_a = utils.RiscoCredito.novo_cliente(2, 1, 0, 2) # boa, alta, nenhuma, >35
pred_a = le_p.inverse_transform(modelo_lr_p.predict(cliente_a))
print(f"6a. Cliente A -> Risco previsto: {pred_a[0]}")

cliente_b = utils.RiscoCredito.novo_cliente(0, 1, 1, 0) # ruim, alta, adequada, <15
pred_b = le_p.inverse_transform(modelo_lr_p.predict(cliente_b))
print(f"6b. Cliente B -> Risco previsto: {pred_b[0]}")

### Questões 7-8: Base 'credit.pkl'

In [ ]:
with open('credit.pkl', 'rb') as f:
    X_train, y_train, X_test, y_test = pickle.load(f)

# 7. Aplicar Regressão Logística
modelo_lr = LogisticRegression(random_state=1)
modelo_lr.fit(X_train, y_train)
y_pred_lr = modelo_lr.predict(X_test)
print(f"7. Taxa de acerto (Acurácia): {accuracy_score(y_test, y_pred_lr)*100:.2f}%")

# 8. Análise de Resultados
print("\n8. Relatório de Classificação:")
print(classification_report(y_test, y_pred_lr))

# PARTE 4: SVM

### 1-3. SVM com Kernel Linear

In [ ]:
# 1. Aplicar algoritmo SVM
# 2. Kernel linear, C=1.0, random_state=1
svm_linear = SVC(kernel='linear', C=1.0, random_state=1)
svm_linear.fit(X_train, y_train)

# 3. Acurácia (Esperado: 0.946)
y_pred_linear = svm_linear.predict(X_test)
print(f"3. Acurácia (Kernel Linear): {accuracy_score(y_test, y_pred_linear):.3f}")

### 4. Teste de demais Kernels

In [ ]:
kernels = ['poly', 'sigmoid', 'rbf']
for k in kernels:
    svm = SVC(kernel=k, C=1.0, random_state=1)
    svm.fit(X_train, y_train)
    acc = accuracy_score(y_test, svm.predict(X_test))
    print(f"Kernel {k:7} -> Acurácia: {acc:.3f}")

print("\nO melhor kernel para esta base de dados é o RBF.")

### 5. Variação do parâmetro C no melhor kernel (RBF)

In [ ]:
valores_c = [1.0, 5.0, 10.0, 50.0, 100.0]
print("Impacto do parâmetro C no kernel RBF:")
print(f"{'C':>8}  {'Acurácia':>10}")
print("-" * 22)
for c in valores_c:
    svm_rbf = SVC(kernel='rbf', C=c, random_state=1)
    svm_rbf.fit(X_train, y_train)
    acc = accuracy_score(y_test, svm_rbf.predict(X_test))
    print(f"{c:>8.1f}  {acc:>10.3f}")

### 6. GridSearch — ajuste automático de hiperparâmetros

In [ ]:
param_grid = {
    'C':      [0.001, 0.01, 0.1, 1, 10, 100],
    'kernel': ['rbf', 'linear', 'poly', 'sigmoid'],
    'gamma':  ['scale', 'auto'],
}

grid_search = GridSearchCV(SVC(random_state=1), param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

print("Melhores hiperparâmetros encontrados pelo GridSearch:")
print(grid_search.best_params_)
print(f"\nAcurácia (validação cruzada): {grid_search.best_score_:.4f}")

y_pred_gs = grid_search.best_estimator_.predict(X_test)
print(f"Acurácia no conjunto de teste:  {accuracy_score(y_test, y_pred_gs):.4f}")

# Comparação ajuste manual vs GridSearch
svm_manual = SVC(kernel='rbf', C=1.0, random_state=1)
svm_manual.fit(X_train, y_train)
acc_manual = accuracy_score(y_test, svm_manual.predict(X_test))
acc_gs     = accuracy_score(y_test, y_pred_gs)
print(f"\nRBF manual (C=1.0): {acc_manual:.4f}")
print(f"GridSearch:         {acc_gs:.4f}")
print(f"Melhor abordagem:   {'GridSearch' if acc_gs >= acc_manual else 'Ajuste manual'}")

# Análise dos resultados dos 4 algoritmos utilizados

### Questão 6 — O SVM é melhor que Naive Bayes, Florestas Aleatórias e Regressão Logística?

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier

# Naive Bayes — a Parte 1 avaliou apenas risco_credito.pkl (14 registros);
# treinamos aqui na base credit.pkl para completar a comparação
nb = GaussianNB()
nb.fit(X_train, y_train)
y_pred_nb = nb.predict(X_test)

# Random Forest — mesmos parâmetros usados na Parte 2 (n_estimators=40, criterion='entropy', random_state=0)
rf = RandomForestClassifier(n_estimators=40, criterion='entropy', random_state=0)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

# Regressão Logística: reutiliza modelo_lr já treinado na questão 7
# SVM RBF C=1: reutiliza o modelo 'svm' treinado na questão 4 (último kernel do loop = rbf)
y_pred_svm_rbf = svm.predict(X_test)

comparacao = {
    'Naive Bayes\n(Parte 1)':          y_pred_nb,
    'Random Forest n=40\n(Parte 2)':   y_pred_rf,
    'Regressão Logística\n(Parte 3)':  y_pred_lr,
    'SVM (RBF, C=1)\n(Parte 3)':       y_pred_svm_rbf,
}

print(f"{'Algoritmo':<30}  {'Acurácia':>9}  {'Precisão (cl.1)':>15}  {'Recall (cl.1)':>13}")
print("-" * 74)
for nome, y_pred in comparacao.items():
    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec  = recall_score(y_test, y_pred, zero_division=0)
    print(f"{nome.replace(chr(10), ' '):<30}  {acc:>9.4f}  {prec:>15.4f}  {rec:>13.4f}")

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for ax, (nome, y_pred) in zip(axes.flat, comparacao.items()):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Não-padrão (0)', 'Padrão (1)'],
                yticklabels=['Não-padrão (0)', 'Padrão (1)'])
    ax.set_title(f"{nome}\nAcurácia: {accuracy_score(y_test, y_pred):.4f}")
    ax.set_xlabel('Predito')
    ax.set_ylabel('Real')
plt.suptitle('Matrizes de Confusão — Comparação dos 4 Algoritmos', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

### Análise escrita

**Métricas observadas (base `credit.pkl`, conjunto de teste com 500 registros):**

| Algoritmo | Parte | Acurácia | Precisão (classe 1) | Recall (classe 1) |
|---|---|---|---|---|
| Naive Bayes | Parte 1* | ~93,8% | ~0,66 | ~0,82 |
| Regressão Logística | Parte 3 | 94,60% | 0,79 | 0,78 |
| Árvore de Decisão | Parte 2 | 98,20% | 0,91 | 0,95 |
| Random Forest (n=40) | Parte 2 | **98,40%** | 0,95 | 0,92 |
| SVM (RBF, C=1) | Parte 3 | 98,20% | ~0,95 | ~0,83 |

> \* A Parte 1 avaliou o Naive Bayes apenas na base `risco_credito.pkl` (14 registros). Os valores acima foram obtidos treinando o mesmo algoritmo (`GaussianNB`) na base `credit.pkl` para permitir a comparação direta. Os valores exatos de NB e SVM são calculados pela célula acima.

**Conclusão:**

O **SVM com kernel RBF** obteve desempenho superior ao **Naive Bayes** e à **Regressão Logística**, e equivalente à **Árvore de Decisão**, ficando marginalmente abaixo do **Random Forest** (diferença de 0,2 p.p.).

- **Naive Bayes (Parte 1)** → acurácia ≈ 93,8%: pior resultado entre os quatro. O modelo assume independência condicional entre os atributos, o que raramente é verdade em dados de crédito. Mesmo assim, o recall da classe de inadimplentes é o mais alto (~0,82), o que significa que ele identifica a maioria dos casos de risco, porém com muitos falsos positivos (baixa precisão ~0,66).

- **Regressão Logística (Parte 3)** → 94,6%: supera o Naive Bayes mas ainda distante dos modelos baseados em árvores e SVM. O modelo é linear — a fronteira de decisão não consegue capturar a não-linearidade dos dados de crédito tão bem quanto os demais. Precisão e recall na classe 1 (0,79 e 0,78) são equilibrados mas modestos.

- **Árvore de Decisão (Parte 2)** → 98,2%: grande salto em relação aos modelos lineares. A árvore aprende regras de corte no espaço dos atributos (renda, idade, empréstimo) de forma não-linear. Recall de 0,95 é o melhor entre todos os modelos, capturando quase todos os inadimplentes.

- **Random Forest n=40 (Parte 2)** → 98,4%: melhor acurácia geral. Ao agregar 40 árvores com amostragem bootstrap e atributos aleatórios, o Random Forest reduz overfitting e aumenta a robustez. Precisão de 0,95 e recall de 0,92 são ambos elevados.

- **SVM (RBF, C=1) (Parte 3)** → 98,2%: empata com a Árvore de Decisão em acurácia. O kernel RBF cria um hiperplano não-linear que separa bem as classes. Após ajuste de hiperparâmetros via GridSearch (questão 6), o SVM atingiu 98,8%, tornando-se o modelo com maior acurácia do trabalho.

**Em resumo:** o SVM RBF supera claramente Naive Bayes e Regressão Logística. Em relação às árvores (Parte 2), os resultados são muito próximos — Árvore de Decisão e SVM empatam em 98,2%, enquanto Random Forest leva uma ligeira vantagem com 98,4%. Com o GridSearch, o SVM se torna competitivo com todos os modelos avaliados neste trabalho.